# VLA Zero-to-Hero: 快速入门教程

本 Notebook 带你从零开始理解 Vision-Language-Action (VLA) 模型的核心概念。

**适用环境**：Google Colab（免费 T4 GPU 可运行）

**你将学到**：
1. 视觉-语言对齐（CLIP）
2. VLA 的基本架构
3. OpenVLA 推理流程
4. Action Chunking 概念
5. 动作可视化方法

## Cell 1: 环境检查和安装

In [ ]:
# Check GPU availability
import torch

print("=" * 50)
print("Environment Check")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU:             {gpu_name}")
    print(f"GPU Memory:      {gpu_mem:.1f} GB")
else:
    print("WARNING: No GPU detected. OpenVLA inference requires GPU.")
    print("  Go to: Runtime -> Change runtime type -> T4 GPU")

print(f"=" * 50)

In [ ]:
# Install dependencies (Colab already has torch)
!pip install -q transformers accelerate pillow matplotlib numpy

## Cell 2: CLIP 视觉-语言对齐

CLIP（Contrastive Language-Image Pre-training）是 VLA 的视觉基础。
它通过对比学习将图像和文本映射到同一向量空间，使得语义相似的图文对距离更近。

下面我们使用预训练 CLIP 计算图像-文本相似度。

In [ ]:
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

from transformers import CLIPProcessor, CLIPModel

# Load CLIP model
print("Loading CLIP model...")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"CLIP loaded on {device}")

In [ ]:
# Create a test image (a red square on a table, simulated with PIL)
def create_test_image(description: str = "table with red block") -> Image.Image:
    """Create a simple synthetic test image for demonstration."""
    img = Image.new("RGB", (224, 224), color=(200, 180, 160))  # beige background
    pixels = img.load()
    # Draw a red square in the center
    for x in range(72, 152):
        for y in range(100, 180):
            pixels[x, y] = (200, 30, 30)  # red
    return img

test_image = create_test_image()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(test_image)
ax.set_title("Test Image: Red Block on Table")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Compute image-text similarity with CLIP
texts = [
    "a red block on a table",          # correct description
    "a blue cup on a table",           # wrong color
    "a robotic arm grasping an object", # wrong scene
    "pick up the red block",           # action-oriented (like VLA instruction)
    "a cat sitting on a sofa",         # completely unrelated
]

inputs = processor(
    text=texts,
    images=test_image,
    return_tensors="pt",
    padding=True,
).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits_per_image  # (1, num_texts)
    probs = logits.softmax(dim=-1)     # normalize to probabilities

# Visualize similarity
probs_np = probs.squeeze(0).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#27ae60" if p == probs_np.max() else "#3498db" for p in probs_np]
bars = ax.barh(range(len(texts)), probs_np, color=colors, edgecolor="white")
ax.set_yticks(range(len(texts)))
ax.set_yticklabels(texts, fontsize=10)
ax.set_xlabel("Similarity Probability", fontsize=11)
ax.set_title("CLIP Image-Text Similarity", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1)
ax.grid(True, alpha=0.3, axis="x", linestyle="--")

# Annotate values
for bar, prob in zip(bars, probs_np):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f"{prob:.3f}", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nKey insight: CLIP correctly identifies the most relevant description.")
print("This alignment capability is the foundation of VLA models.")

In [ ]:
# Visualize CLIP attention map (which image regions the model focuses on)
def get_clip_attention(image: Image.Image, model, processor, device) -> np.ndarray:
    """Extract CLIP vision transformer attention and compute rollout."""
    inputs = processor(images=image, return_tensors="pt").to(device)
    
    attn_maps = []
    
    def hook_fn(module, input, output):
        # output[1] is attention weights: (batch, heads, seq_len, seq_len)
        if output[1] is not None:
            attn_maps.append(output[1].detach().cpu())
    
    hooks = []
    for layer in model.vision_model.encoder.layers:
        h = layer.self_attn.register_forward_hook(hook_fn)
        hooks.append(h)
    
    with torch.no_grad():
        _ = model.get_image_features(**inputs)
    
    for h in hooks:
        h.remove()
    
    if not attn_maps:
        return np.ones((7, 7))  # fallback uniform map
    
    # Attention rollout
    rollout = torch.eye(attn_maps[0].shape[-1])
    for attn in attn_maps:
        attn_avg = attn.mean(dim=1)[0, 0:1, :]  # CLS token attention
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)
        rollout = torch.matmul(attn_avg, rollout)
    
    # Remove CLS token, reshape to grid
    grid_size = int(np.sqrt(rollout.shape[-1] - 1))
    attn_map = rollout[0, 1:].reshape(grid_size, grid_size).numpy()
    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map

attn_map = get_clip_attention(test_image, model, processor, device)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original image
axes[0].imshow(test_image)
axes[0].set_title("Original Image", fontweight="bold")
axes[0].axis("off")

# Attention heatmap
im = axes[1].imshow(attn_map, cmap="hot", interpolation="bilinear")
axes[1].set_title("CLIP Attention Map", fontweight="bold")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

# Overlay
axes[2].imshow(test_image)
axes[2].imshow(attn_map, cmap="jet", alpha=0.5, interpolation="bilinear")
axes[2].set_title("Attention Overlay", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("\nThe attention map shows which image regions CLIP focuses on.")
print("In a VLA, this visual understanding guides action prediction.")

## Cell 3: 最小 VLA 架构搭建

VLA = 视觉编码器 + 语言编码器 + 融合层 + 动作头

这里我们用纯 PyTorch 从零搭建一个最小 VLA，理解其架构。

In [ ]:
import torch
import torch.nn as nn


class MinimalVLA(nn.Module):
    """
    Minimal VLA architecture:
      Vision Encoder (CNN) -> Text Encoder (Embedding + GRU) ->
      Fusion (concat + MLP) -> Policy Head (MLP -> 7-DOF action)
    
    In production, replace with:
      Vision: CLIP / DINOv2
      Language: LLaMA / T5
      Fusion: Cross-attention
    """
    
    def __init__(self, image_size=224, action_dim=7, embed_dim=256):
        super().__init__()
        self.action_dim = action_dim
        
        # Vision encoder: simple CNN
        self.vision_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),    # 112x112
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),   # 56x56
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # 28x28
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, embed_dim),
        )
        
        # Text encoder: embedding + GRU
        self.text_embedding = nn.Embedding(1000, embed_dim)
        self.text_encoder = nn.GRU(embed_dim, embed_dim, batch_first=True)
        
        # Fusion: concat + MLP
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        # Policy head: MLP -> action
        self.policy_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Tanh(),  # output in [-1, 1]
        )
    
    def forward(self, image, text_tokens):
        """
        Args:
            image: [B, 3, H, W]
            text_tokens: [B, seq_len]
        Returns:
            action: [B, action_dim] in [-1, 1]
        """
        vis_feat = self.vision_encoder(image)              # [B, embed_dim]
        emb = self.text_embedding(text_tokens)            # [B, seq_len, embed_dim]
        _, hidden = self.text_encoder(emb)                # [1, B, embed_dim]
        txt_feat = hidden.squeeze(0)                      # [B, embed_dim]
        
        fused = torch.cat([vis_feat, txt_feat], dim=-1)   # [B, embed_dim*2]
        fused = self.fusion(fused)                        # [B, embed_dim]
        
        action = self.policy_head(fused)                  # [B, action_dim]
        return action


# Simple tokenizer for demo
def simple_tokenize(text, max_len=16):
    tokens = [ord(c) % 1000 for c in text.lower()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))
    return torch.tensor(tokens, dtype=torch.long).unsqueeze(0)


print("MinimalVLA architecture defined!")
print(f"Parameters: {sum(p.numel() for p in MinimalVLA().parameters()):,}")
print(f"  (Compare: OpenVLA-7B has ~7 billion parameters)")

In [ ]:
# Forward pass demonstration
model = MinimalVLA(action_dim=7, embed_dim=256)
model.eval()

# Create dummy inputs
dummy_image = torch.randn(1, 3, 224, 224)
instruction = "pick up the red block"
text_tokens = simple_tokenize(instruction)

# Forward pass
with torch.no_grad():
    action = model(dummy_image, text_tokens)

# Display results
action_names = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]
action_np = action.squeeze().numpy()

print(f"\nInput: '{instruction}'")
print(f"Output action (raw, [-1, 1]):")
print("-" * 35)
for name, val in zip(action_names, action_np):
    print(f"  {name:12s}: {val:+.4f}")
print("-" * 35)

# Visualize as bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#e74c3c" if abs(v) > 0.5 else "#3498db" for v in action_np]
ax.bar(action_names, action_np, color=colors, edgecolor="white", width=0.6)
ax.axhline(y=0, color="gray", linewidth=0.5)
ax.set_ylabel("Action Value")
ax.set_title(f"MinimalVLA Output (Random Init) -- '{instruction}'",
             fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nNOTE: This model is randomly initialized, so the output is meaningless.")
print("After training on robot data, these values would represent actual robot commands.")

In [ ]:
# Architecture diagram
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis("off")
ax.set_title("VLA Architecture Pipeline", fontsize=14, fontweight="bold")

# Draw boxes
boxes = [
    (0.5, 1.5, "Image\n224x224", "#e74c3c"),
    (2.5, 1.5, "Vision\nEncoder\n(CNN/CLIP)", "#e67e22"),
    (4.8, 1.5, "Fusion\nLayer\n(Concat/Attn)", "#f1c40f"),
    (7.0, 1.5, "Policy\nHead\n(MLP)", "#27ae60"),
    (9.2, 1.5, "Action\n7-DOF", "#3498db"),
    (2.5, 0.0, "Text\nEncoder\n(LLaMA/T5)", "#9b59b6"),
]

for x, y, text, color in boxes:
    rect = plt.Rectangle((x, y), 2.0, 1.2, linewidth=2,
                          edgecolor=color, facecolor=color, alpha=0.2)
    ax.add_patch(rect)
    ax.text(x + 1.0, y + 0.6, text, ha="center", va="center",
            fontsize=9, fontweight="bold")

# Draw arrows
arrows = [
    (2.5, 2.1, 4.8, 2.1),   # vision -> fusion
    (3.5, 1.5, 4.8, 1.5),   # text -> fusion (diagonal up-right)
    (6.8, 2.1, 7.0, 2.1),   # fusion -> policy
    (9.0, 2.1, 9.2, 2.1),   # policy -> action
]

for x1, y1, x2, y2 in arrows:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="gray", lw=2))

plt.tight_layout()
plt.show()

## Cell 4: OpenVLA 推理（需 GPU）

> **需要 GPU（Colab 免费 T4 可运行，但较慢）**
>
> OpenVLA-7B 有约 70 亿参数，需要约 16GB 显存。
> 在 Colab T4（16GB）上勉强可以运行，推理速度约 1-2 秒/步。
> 如果显存不足，可以尝试使用 4-bit 量化加载。

OpenVLA 是目前最流行的开源 VLA 模型之一，基于 LLaMA 架构，
将 CLIP 视觉编码器与 LLaMA 语言模型结合，直接输出机器人动作。

In [ ]:
# Check if we have enough GPU memory
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU Memory: {gpu_mem_gb:.1f} GB")
    if gpu_mem_gb < 14:
        print("WARNING: < 14GB VRAM. Consider using 4-bit quantization.")
        print("  Set use_4bit = True below.")
    else:
        print("Sufficient GPU memory for OpenVLA-7B (bf16).")
else:
    print("ERROR: GPU required for OpenVLA. Skipping this cell.")
    print("  Go to: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Load OpenVLA (uncomment to run - takes ~3 min on first load)
load_openvla = False  # Set to True to actually load the model

if load_openvla and torch.cuda.is_available():
    from transformers import AutoModelForVision2Seq, AutoProcessor
    
    use_4bit = torch.cuda.get_device_properties(0).total_mem / 1024**3 < 14
    
    print("Loading OpenVLA-7B... (this takes a few minutes)")
    
    if use_4bit:
        print("Using 4-bit quantization to save memory...")
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        ovla_model = AutoModelForVision2Seq.from_pretrained(
            "openvla/openvla-7b",
            quantization_config=bnb_config,
            trust_remote_code=True,
        )
    else:
        ovla_model = AutoModelForVision2Seq.from_pretrained(
            "openvla/openvla-7b",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        )
    
    ovla_processor = AutoProcessor.from_pretrained(
        "openvla/openvla-7b", trust_remote_code=True
    )
    ovla_model = ovla_model.to("cuda")
    ovla_model.eval()
    
    print("OpenVLA loaded successfully!")
    
    # Count parameters
    total_params = sum(p.numel() for p in ovla_model.parameters())
    print(f"Total parameters: {total_params:,} (~{total_params/1e9:.1f}B)")
else:
    print("Skipping OpenVLA loading (load_openvla=False or no GPU).")
    print("\nHow OpenVLA works (without loading):")
    print("  1. Input: Image (224x224) + Text instruction")
    print("  2. CLIP ViT-L/14 encodes image -> visual tokens")
    print("  3. LLaMA-7B processes visual tokens + text prompt")
    print("  4. Output: 7-DOF action token (normalized, then unnormalized)")
    print("  5. The action is directly executable on the robot")
    print("\nPrompt format:")
    print('  "In: What action should the robot take to {instruction}?\nOut:"')

In [ ]:
# Run OpenVLA inference (uncomment if model is loaded)
if load_openvla and torch.cuda.is_available() and 'ovla_model' in dir():
    # Use our test image
    instruction = "pick up the red block"
    prompt = f"In: What action should the robot take to {instruction}?\nOut:"
    
    inputs = ovla_processor(prompt, test_image, return_tensors="pt").to("cuda")
    
    import time
    start = time.time()
    with torch.no_grad():
        action = ovla_model.predict_action(**inputs, unnorm_key="bridge")
    elapsed = time.time() - start
    
    print(f"Inference time: {elapsed:.2f}s")
    print(f"\nPredicted action (unnormalized):")
    action_names = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]
    for name, val in zip(action_names, action):
        print(f"  {name:12s}: {val:+.6f}")
    
    # Interpret
    pos_mag = sum(x**2 for x in action[:3])**0.5
    print(f"\n  Position delta: {pos_mag:.4f} m")
    print(f"  Gripper: {'OPEN' if action[6] > 0 else 'CLOSE'}")
else:
    print("Skipping inference demo.")
    print("\n--- Expected Output Format ---")
    print("  dx:       +0.0234   (move 2.3cm in X)")
    print("  dy:       -0.0156   (move 1.6cm in -Y)")
    print("  dz:       -0.0312   (move 3.1cm down)")
    print("  droll:    +0.0012   (tiny roll adjustment)")
    print("  dpitch:   -0.0008   (tiny pitch adjustment)")
    print("  dyaw:     +0.0015   (tiny yaw adjustment)")
    print("  gripper:  -0.0500   (close gripper)")

## Cell 5: Action Chunking 实现

**Action Chunking** 是 VLA 推理中非常重要的技术：
- **单步模式**：每步都重新推理，延迟高但灵活
- **Chunk 模式**：一次推理输出多个动作，减少推理次数

ACT (Action Chunking with Transformers) 和 Diffusion Policy 都使用这种技术。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class ActionChunker:
    """
    Action Chunking buffer.
    
    Instead of querying the policy every step, we:
    1. Query the policy once to get a chunk of K actions
    2. Execute actions from the buffer one by one
    3. When the buffer is exhausted, query again
    
    Benefits:
    - Reduces inference frequency (Kx fewer model calls)
    - Smoother trajectories (temporal consistency within chunk)
    - Lower latency
    
    Trade-off:
    - Less reactive to environmental changes
    - May execute outdated actions if scene changes
    """
    
    def __init__(self, chunk_size: int = 10, action_dim: int = 7):
        self.chunk_size = chunk_size
        self.action_dim = action_dim
        self.buffer = []
        self.buffer_idx = 0
    
    def is_empty(self) -> bool:
        return len(self.buffer) == 0 or self.buffer_idx >= len(self.buffer)
    
    def get_action(self) -> np.ndarray:
        """Get next action from buffer."""
        if self.is_empty():
            raise RuntimeError("Buffer empty! Call fill_buffer() first.")
        action = self.buffer[self.buffer_idx]
        self.buffer_idx += 1
        return action
    
    def fill_buffer(self, policy_fn, observation, instruction):
        """Query policy to fill the action buffer."""
        # In a real VLA, the policy would output a chunk directly.
        # Here we simulate by querying K times with slight noise.
        self.buffer = []
        for i in range(self.chunk_size):
            action = policy_fn(observation, instruction)
            # Add temporal smoothing: later actions have slightly less magnitude
            decay = 1.0 - 0.03 * i
            self.buffer.append(action * decay)
        self.buffer_idx = 0


print("ActionChunker defined!")
print(f"  Chunk size: 10 actions per inference")
print(f"  With 10Hz control: 1 inference per second instead of 10")

In [ ]:
# Compare: single-step vs chunked execution
np.random.seed(42)
total_steps = 100

# Simulated policy (smooth sinusoidal + noise)
def policy_fn(obs, instruction):
    t = len(single_step_actions)  # use step count for deterministic output
    base = np.sin(t * 0.1) * 0.3
    noise = np.random.randn(7) * 0.1
    action = np.zeros(7)
    action[0] = base + noise[0]  # dx follows a smooth pattern
    action[2] = -0.02             # constant downward
    action[6] = 0.5              # gripper half-open
    return action

# Single-step: query every step
single_step_actions = []
inference_count_single = 0
for step in range(total_steps):
    action = policy_fn(None, "pick")
    single_step_actions.append(action)
    inference_count_single += 1

# Chunked: query every K steps
chunk_size = 10
chunker = ActionChunker(chunk_size=chunk_size, action_dim=7)
chunked_actions = []
inference_count_chunked = 0

# Need a counter for the chunked policy
chunk_step_counter = [0]
def chunked_policy_fn(obs, instruction):
    t = chunk_step_counter[0]
    chunk_step_counter[0] += 1
    base = np.sin(t * 0.1) * 0.3
    noise = np.random.randn(7) * 0.1
    action = np.zeros(7)
    action[0] = base + noise[0]
    action[2] = -0.02
    action[6] = 0.5
    return action

for step in range(total_steps):
    if chunker.is_empty():
        chunker.fill_buffer(chunked_policy_fn, None, "pick")
        inference_count_chunked += 1
    action = chunker.get_action()
    chunked_actions.append(action)

single_step_actions = np.array(single_step_actions)
chunked_actions = np.array(chunked_actions)

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Single-Step vs Action Chunking", fontsize=14, fontweight="bold")

# dx comparison
axes[0, 0].plot(single_step_actions[:, 0], alpha=0.7, label="Single-step", linewidth=0.8)
axes[0, 0].plot(chunked_actions[:, 0], alpha=0.7, label=f"Chunked (K={chunk_size})", linewidth=1.2)
axes[0, 0].set_title("dx (X position delta)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Smoothness comparison (rolling std)
window = 5
single_smooth = np.array([np.std(single_step_actions[max(0,i-window):i+1, 0]) 
                           for i in range(total_steps)])
chunk_smooth = np.array([np.std(chunked_actions[max(0,i-window):i+1, 0]) 
                          for i in range(total_steps)])
axes[0, 1].plot(single_smooth, alpha=0.7, label="Single-step")
axes[0, 1].plot(chunk_smooth, alpha=0.7, label=f"Chunked (K={chunk_size})")
axes[0, 1].set_title("Local Smoothness (rolling std of dx)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gripper comparison
axes[1, 0].plot(single_step_actions[:, 6], alpha=0.7, label="Single-step")
axes[1, 0].plot(chunked_actions[:, 6], alpha=0.7, label=f"Chunked (K={chunk_size})")
axes[1, 0].set_title("Gripper state")
axes[1, 0].set_ylim(-0.1, 1.1)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Inference count bar chart
methods = ["Single-Step", f"Chunked (K={chunk_size})"]
counts = [inference_count_single, inference_count_chunked]
colors = ["#e74c3c", "#27ae60"]
bars = axes[1, 1].bar(methods, counts, color=colors, edgecolor="white", width=0.5)
for bar, count in zip(bars, counts):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    str(count), ha="center", fontweight="bold", fontsize=12)
axes[1, 1].set_ylabel("Number of Inferences")
axes[1, 1].set_title(f"Inference Efficiency ({total_steps} steps)")
axes[1, 1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"\nKey takeaway:")
print(f"  Single-step: {inference_count_single} inferences for {total_steps} steps")
print(f"  Chunked:     {inference_count_chunked} inferences for {total_steps} steps")
print(f"  Reduction:   {inference_count_single / max(inference_count_chunked, 1):.1f}x fewer inferences")

## Cell 6: 动作可视化

理解 VLA 输出的动作向量的含义和可视化方法。

In [ ]:
# Generate a realistic-looking action trajectory for visualization
np.random.seed(123)
num_steps = 150

# Simulate a pick-and-place trajectory
actions = np.zeros((num_steps, 7))

# Phase 1: Move above object (steps 0-40)
t1 = np.linspace(0, 1, 40)
actions[:40, 0] = 0.05 * np.sin(t1 * np.pi)  # dx: move right then stop
actions[:40, 1] = -0.03 * t1                   # dy: move forward
actions[:40, 2] = 0.02 * np.ones(40)           # dz: go up
actions[:40, 6] = 1.0                           # gripper open

# Phase 2: Descend (steps 40-70)
t2 = np.linspace(0, 1, 30)
actions[40:70, 2] = -0.03 * np.ones(30)        # dz: go down
actions[40:70, 6] = 1.0                         # gripper still open

# Phase 3: Grip (steps 70-85)
t3 = np.linspace(0, 1, 15)
actions[70:85, 6] = 1.0 - t3                   # gripper closing

# Phase 4: Lift (steps 85-120)
t4 = np.linspace(0, 1, 35)
actions[85:120, 2] = 0.03 * np.ones(35)        # dz: go up
actions[85:120, 0] = -0.02 * t4                 # dx: move back
actions[85:120, 6] = 0.0                         # gripper closed

# Phase 5: Place (steps 120-150)
t5 = np.linspace(0, 1, 30)
actions[120:150, 2] = -0.02 * t5                # dz: go down to place
actions[120:150, 6] = t5                         # gripper opening

# Add small noise
actions += np.random.randn(*actions.shape) * 0.002

# Phase labels
phase_labels = ([-40, 70, 85, 120] + [0] * (num_steps - 4))[:num_steps]

print(f"Generated trajectory: {num_steps} steps, 7-DOF")
print(f"Simulated: move above -> descend -> grip -> lift -> place")

In [ ]:
# Full action trajectory visualization
action_names = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]
action_colors = ["#e74c3c", "#27ae60", "#3498db", "#9b59b6", "#e67e22", "#1abc9c", "#2c3e50"]

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)

# --- All 7-DOF actions ---
ax_all = fig.add_subplot(gs[0, :])
for i, (name, color) in enumerate(zip(action_names, action_colors)):
    lw = 2.0 if name == "gripper" else 0.8
    ax_all.plot(actions[:, i], label=name, color=color, linewidth=lw, alpha=0.8)

# Phase background colors
phase_boundaries = [0, 40, 70, 85, 120, 150]
phase_names = ["Move Above", "Descend", "Grip", "Lift", "Place"]
phase_colors = ["#e74c3c", "#f39c12", "#9b59b6", "#27ae60", "#3498db"]
for i in range(len(phase_names)):
    ax_all.axvspan(phase_boundaries[i], phase_boundaries[i+1],
                   alpha=0.06, color=phase_colors[i])
    mid = (phase_boundaries[i] + phase_boundaries[i+1]) / 2
    ax_all.text(mid, ax_all.get_ylim()[1] if i == 0 else 0.08,
                phase_names[i], ha="center", va="top", fontsize=8,
                color=phase_colors[i], fontweight="bold")

ax_all.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
ax_all.set_xlabel("Step")
ax_all.set_ylabel("Action Value")
ax_all.set_title("7-DOF Action Trajectory (Simulated Pick-and-Place)",
                  fontsize=12, fontweight="bold")
ax_all.legend(loc="upper right", ncol=7, fontsize=8)
ax_all.grid(True, alpha=0.2)

# --- Position actions (dx, dy, dz) ---
ax_pos = fig.add_subplot(gs[1, 0])
for i, (name, color) in enumerate(zip(action_names[:3], action_colors[:3])):
    ax_pos.plot(actions[:, i], label=name, color=color, linewidth=1.2)
ax_pos.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
ax_pos.set_xlabel("Step")
ax_pos.set_ylabel("Delta Position (m)")
ax_pos.set_title("Position Actions", fontweight="bold")
ax_pos.legend()
ax_pos.grid(True, alpha=0.3)

# --- Gripper ---
ax_grip = fig.add_subplot(gs[1, 1])
ax_grip.fill_between(range(num_steps), actions[:, 6], alpha=0.3, color="steelblue")
ax_grip.plot(actions[:, 6], color="steelblue", linewidth=1.5)
ax_grip.set_xlabel("Step")
ax_grip.set_ylabel("Gripper (0=close, 1=open)")
ax_grip.set_title("Gripper State", fontweight="bold")
ax_grip.set_ylim(-0.1, 1.15)
ax_grip.grid(True, alpha=0.3)

# Mark open/close transitions
grip_diff = np.diff(actions[:, 6])
close_pts = np.where(grip_diff < -0.15)[0]
open_pts = np.where(grip_diff > 0.15)[0]
if len(close_pts) > 0:
    ax_grip.scatter(close_pts, actions[close_pts, 6], c="red", s=50, zorder=5, label="Close")
if len(open_pts) > 0:
    ax_grip.scatter(open_pts, actions[open_pts, 6], c="green", s=50, zorder=5, label="Open")
ax_grip.legend()

# --- Cumulative displacement ---
ax_disp = fig.add_subplot(gs[2, 0])
cumulative_pos = np.cumsum(actions[:, :3], axis=0)
labels_pos = ["X", "Y", "Z"]
for i, (label, color) in enumerate(zip(labels_pos, action_colors[:3])):
    ax_disp.plot(cumulative_pos[:, i], label=f"Cumulative {label}", color=color, linewidth=1.2)
ax_disp.set_xlabel("Step")
ax_disp.set_ylabel("Cumulative Position (m)")
ax_disp.set_title("Cumulative Displacement", fontweight="bold")
ax_disp.legend()
ax_disp.grid(True, alpha=0.3)

# --- Action magnitude over time ---
ax_mag = fig.add_subplot(gs[2, 1])
pos_magnitude = np.linalg.norm(actions[:, :3], axis=1)
rot_magnitude = np.linalg.norm(actions[:, 3:6], axis=1)
ax_mag.plot(pos_magnitude, label="Position ||a_pos||", color="#e74c3c", linewidth=1.2)
ax_mag.plot(rot_magnitude, label="Rotation ||a_rot||", color="#9b59b6", linewidth=1.2, alpha=0.7)
ax_mag.set_xlabel("Step")
ax_mag.set_ylabel("L2 Norm")
ax_mag.set_title("Action Magnitude", fontweight="bold")
ax_mag.legend()
ax_mag.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3D trajectory visualization
from mpl_toolkits.mplot3d import Axes3D

cumulative = np.cumsum(actions[:, :3], axis=0)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")

# Color by step
scatter = ax.scatter(
    cumulative[:, 0], cumulative[:, 1], cumulative[:, 2],
    c=np.arange(num_steps), cmap="viridis", s=15, alpha=0.7,
)
ax.plot(cumulative[:, 0], cumulative[:, 1], cumulative[:, 2],
        "gray", linewidth=0.5, alpha=0.5)

# Mark start and end
ax.scatter(*cumulative[0], c="green", s=150, marker="o", label="Start", zorder=5)
ax.scatter(*cumulative[-1], c="red", s=150, marker="*", label="End", zorder=5)

# Mark key events
ax.scatter(*cumulative[70], c="purple", s=100, marker="D", label="Grip start", zorder=5)
ax.scatter(*cumulative[120], c="orange", s=100, marker="s", label="Place start", zorder=5)

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title("3D End-Effector Trajectory (Cumulative Actions)", fontweight="bold")
ax.legend()

plt.colorbar(scatter, ax=ax, label="Step", shrink=0.6, pad=0.1)
plt.tight_layout()
plt.show()

print("\nThis 3D trajectory shows the robot's end-effector path")
print("during a simulated pick-and-place task.")
print("Color indicates time progression (green=early, purple=late).")

## Cell 7: 下一步指引

完成本 Notebook 后你已经掌握：
- **VLM 的视觉-语言对齐原理** -- CLIP 如何将图像和文本映射到同一空间
- **VLA 的基本架构** -- 视觉编码器 + 语言编码器 + 融合层 + 动作头
- **OpenVLA 推理流程** -- 从图像+指令到机器人动作的端到端流程
- **Action Chunking 概念** -- 如何通过动作分块提升推理效率
- **动作可视化方法** -- 理解不同动作表示的含义

### 接下来：

1. **本地运行** `tutorials/02-action-representation/` -- 深入理解动作空间设计
2. **运行仿真闭环** `examples/sim_closed_loop_demo.py` -- 看到完整的 observe-think-act 循环
3. **微调训练** `tutorials/04-fine-tuning/finetune_libero.py` -- 在真实任务数据上训练 VLA

### 推荐学习路径：

| 阶段 | 内容 | 文件 |
|------|------|------|
| 基础 | VLA 概念 | `docs/01-what-is-vla.md` |
| 基础 | 关键论文 | `docs/02-key-papers.md` |
| 进阶 | 动作表示 | `tutorials/02-action-representation/` |
| 进阶 | 简单 VLA | `tutorials/03-simple-vla/` |
| 高级 | 微调训练 | `tutorials/04-fine-tuning/` |
| 实践 | 仿真闭环 | `examples/sim_closed_loop_demo.py` |
| 实践 | 可视化 | `examples/visualize_vla.py` |

### 关键资源：
- [OpenVLA GitHub](https://github.com/openvla/openvla)
- [Octo GitHub](https://github.com/octo-models/octo)
- [LIBERO Benchmark](https://libero-project.github.io/)
- [RLDS Dataset Format](https://github.com/google-research/rlds)